In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 12.8 MB/s eta 0:00:00


In [2]:
def calc_quote_skew(
    inventory,        # 当前库存（BTC）
    target_inventory, # 目标库存（BTC）
    max_inventory,    # 最大允许库存
    imbalance,        # 订单簿不平衡度 0~1
    volatility,       # 近期波动率（百分比）
    base_spread_bps,  # 基础价差（基点）
    funding_rate      # 资金费率
):
    # 1. 库存偏离度信号
    inv_deviation = (inventory - target_inventory) / target_inventory
    # 范围：-1（库存空了）~ +1（库存爆了）

    # 2. Imbalance信号
    # imbalance > 0.55 = 买方强，< 0.45 = 卖方强
    imbalance_signal = (imbalance - 0.5) * 2  # 转成 -1~1

    # 3. 波动率调整价差
    vol_adjustment = 1 + (volatility * 10)  # 波动率每1%价差扩大10%

    # 4. 资金费率方向（正=多头过热，可能跌）
    funding_signal = funding_rate * 100  # 转成可用数值

    # 综合偏移量（定价偏移百分比）
    # 正=往上偏（看涨报价），负=往下偏（看跌报价）
    skew = (
        -0.6 * inv_deviation +       # 库存多→往下偏（降价卖）
        -0.3 * imbalance_signal +     # 买方强→报价往上偏
        -0.1 * funding_signal         # 资金费率高→报价往下偏
    )

    # 调整后的价差
    spread = base_spread_bps * vol_adjustment

    # 计算实际报价
    mid_price = 63500  # 当前中间价
    bid = mid_price * (1 - spread / 10000 / 2 + skew / 10000)
    ask = mid_price * (1 + spread / 10000 / 2 + skew / 10000)

    return {
        "skew": round(skew, 2),
        "spread_bps": round(spread, 1),
        "bid": round(bid, 1),
        "ask": round(ask, 1),
        "inventory_signal": round(-0.6 * inv_deviation, 3),
        "imbalance_signal": round(-0.3 * imbalance_signal, 3),
        "funding_signal": round(-0.1 * funding_signal, 3),
    }

# 跑一下
result = calc_quote_skew(
    inventory=15,           # 手里15个BTC（超标了）
    target_inventory=10,    # 目标10个
    max_inventory=20,
    imbalance=0.58,         # 买方偏强
    volatility=0.02,        # 近24h波动2%
    base_spread_bps=3.0,    # BTC基础价差3bps
    funding_rate=0.0001     # 资金费率0.01%（正常）
)

print(f"综合偏移: {result['skew']} bps")
print(f"调整后价差: {result['spread_bps']} bps")
print(f"库存信号贡献: {result['inventory_signal']}")
print(f"Imbalance信号贡献: {result['imbalance_signal']}")
print(f"资金费率信号贡献: {result['funding_signal']}")
print(f"报价 买: ${result['bid']} | 卖: ${result['ask']}")

综合偏移: -0.35 bps
调整后价差: 3.6 bps
库存信号贡献: -0.3
Imbalance信号贡献: -0.048
资金费率信号贡献: -0.001
报价 买: $63486.4 | 卖: $63509.2
